# Test the deployed AgentCore Runtime

This notebook invokes the **deployed** Amazon Bedrock AgentCore Runtime from stack `00-agent-agentcore` with `InvokeAgentRuntime`, the same call the Lambda functions of stacks 01 and 02 make.

It covers text, short-term memory, image, audio transcript, video, document, a rejected payload, and a look at long-term memory records.

Requirements:

* Stack `00-agent-agentcore` deployed in your account and Region.
* AWS credentials that can read the `/agentcore/*` SSM parameters, call `bedrock-agentcore:InvokeAgentRuntime` on the runtime and `bedrock-agentcore:RetrieveMemoryRecords` on the memory.
* `boto3` 1.43 or later. The `bedrock-agentcore` data plane is part of the public SDK, so no custom botocore model is needed.

Outputs are not stored in this notebook. Run the cells in order.

In [ ]:
%pip install "boto3>=1.43" --quiet

## 1. Setup: read the stack outputs from SSM

In [ ]:
import base64
import json
import os
import time
import uuid
from pathlib import Path

import boto3

REGION = os.environ.get("AWS_REGION") or boto3.Session().region_name
if not REGION:
    raise RuntimeError("Set AWS_REGION to the Region where stack 00 is deployed.")

ssm = boto3.client("ssm", region_name=REGION)


def get_param(name: str) -> str:
    return ssm.get_parameter(Name=name)["Parameter"]["Value"]


AGENT_RUNTIME_ARN = get_param("/agentcore/agent_runtime_arn")
S3_BUCKET = get_param("/agentcore/s3_bucket_name")
MEMORY_ID = get_param("/agentcore/memory_id")

print("Region:    ", REGION)
print("Runtime:   ", AGENT_RUNTIME_ARN.split("/")[-1])
print("Memory ID: ", MEMORY_ID)
print("S3 bucket: ", S3_BUCKET)

## 2. Helper: invoke the runtime

Two identifiers drive memory. They must be different strings and at least 33 characters long (the Lambdas pad them with `ljust(33, "0")`):

* `actor_id` identifies the **user**. It drives long-term memory (facts, preferences, summaries).
* `session_id` identifies the **conversation**. It is sent as `runtimeSessionId` and drives short-term memory.

The payload carries `actor_id` and an optional `idempotency_key`. The runtime keeps a small cache of recent keys, so a retried request with the same key returns the stored answer instead of calling the model again.

In [ ]:
agentcore = boto3.client("bedrock-agentcore", region_name=REGION)

TEST_PHONE = "5730012345670000000000"
ACTOR_ID = f"wa-user-{TEST_PHONE}".ljust(33, "0")
SESSION_ID = f"wa-chat-{uuid.uuid4().hex}".ljust(33, "0")  # new conversation on every run
print(f"actor_id:   {ACTOR_ID} ({len(ACTOR_ID)} chars)")
print(f"session_id: {SESSION_ID} ({len(SESSION_ID)} chars)")


def invoke_agent(prompt: str, media: dict | None = None, idempotency_key: str | None = None) -> str:
    """Invoke the deployed runtime and return the reply text."""
    payload = {
        "prompt": prompt,
        "actor_id": ACTOR_ID,
        "idempotency_key": idempotency_key or uuid.uuid4().hex,
    }
    if media:
        payload["media"] = media

    response = agentcore.invoke_agent_runtime(
        agentRuntimeArn=AGENT_RUNTIME_ARN,
        runtimeSessionId=SESSION_ID,
        payload=json.dumps(payload).encode("utf-8"),
    )
    body = response["response"].read().decode("utf-8")
    try:
        return json.loads(body).get("result", body)
    except json.JSONDecodeError:
        return body


print("Helper ready.")

## 3. Text message

In [ ]:
print(invoke_agent("Hola! Mi nombre es Carlos y soy de Colombia. ¿Qué puedes hacer por mí?"))

## 4. Short-term memory (same session)

The agent should remember the name and the country from the previous turn.

In [ ]:
print(invoke_agent("¿Cómo me llamo y de dónde soy?"))

## 5. Idempotency

Sending the same `idempotency_key` twice returns the cached answer for the second call. This is what protects you from duplicate model calls when Meta or Lambda retries a delivery.

In [ ]:
key = uuid.uuid4().hex
first = invoke_agent("Dime un dato curioso sobre Colombia.", idempotency_key=key)
second = invoke_agent("Dime un dato curioso sobre Colombia.", idempotency_key=key)
print("Same answer for the retried key:", first == second)

## 6. Image

The image is sent as base64. The runtime limits decoded media to about 1.5 MB, so large photos are rejected with a friendly message. The test image in this folder is larger than that, so the cell below shrinks it first when Pillow is available.

In [ ]:
IMAGE_PATH = Path("imagen2.png")
MAX_BYTES = 1_400_000

image_bytes = IMAGE_PATH.read_bytes() if IMAGE_PATH.exists() else None
fmt = "png"
if image_bytes and len(image_bytes) > MAX_BYTES:
    try:
        import io
        from PIL import Image

        img = Image.open(io.BytesIO(image_bytes)).convert("RGB")
        img.thumbnail((1568, 1568))
        buf = io.BytesIO()
        img.save(buf, format="JPEG", quality=85)
        image_bytes, fmt = buf.getvalue(), "jpeg"
    except ImportError:
        print("Install Pillow (%pip install pillow) to shrink the image, or use a smaller one.")
        image_bytes = None

if image_bytes:
    print(f"Sending {len(image_bytes) / 1_000_000:.2f} MB as {fmt}")
    print(invoke_agent(
        "Describe esta imagen en detalle.",
        media={"type": "image", "format": fmt, "data": base64.b64encode(image_bytes).decode()},
    ))
else:
    print("No usable image found. Put a small PNG or JPEG next to this notebook.")

## 7. Audio transcript

In production the Lambda transcribes voice notes with Amazon Transcribe and sends the text. This cell sends a transcript directly.

In [ ]:
print(invoke_agent(
    "Procesa este audio.",
    media={
        "type": "audio_transcript",
        "data": "Necesito agendar una reunión para el martes a las 3 de la tarde con el equipo de marketing para hablar de la estrategia del cuarto trimestre.",
    },
))

## 8. Video (TwelveLabs)

Requires the TwelveLabs API key in the secret created by stack 00 (see the stack README). Set `VIDEO_PATH` to a local MP4. The file is uploaded under the `video/` prefix, which is where the runtime role can read.

In [ ]:
VIDEO_PATH = Path("sample_video.mp4")  # bring your own short MP4

if VIDEO_PATH.exists():
    key = f"video/{VIDEO_PATH.name}"
    boto3.client("s3", region_name=REGION).upload_file(str(VIDEO_PATH), S3_BUCKET, key)
    print(invoke_agent("Analiza este video en detalle.", media={"type": "video", "s3_uri": f"s3://{S3_BUCKET}/{key}"}))
else:
    print(f"{VIDEO_PATH} not found, skipping the video test.")

## 9. Document

In [ ]:
DOCUMENT_PATH = Path("sample_document.pdf")  # bring your own small PDF (under 1.5 MB)

if DOCUMENT_PATH.exists():
    print(invoke_agent(
        "Resume este documento.",
        media={
            "type": "document",
            "format": DOCUMENT_PATH.suffix.lstrip("."),
            "data": base64.b64encode(DOCUMENT_PATH.read_bytes()).decode(),
            "name": DOCUMENT_PATH.stem,
        },
    ))
else:
    print(f"{DOCUMENT_PATH} not found, skipping the document test.")

## 10. A rejected payload

Unsupported formats never reach the model or the memory. The runtime answers with a short message.

In [ ]:
print(invoke_agent("mira esto", media={"type": "image", "format": "bmp", "data": "AAAA"}))

## 11. Cross-turn summary

In [ ]:
print(invoke_agent("Dame un resumen de todo lo que hemos hablado en esta conversación."))

## 12. Inspect long-term memory

AgentCore extracts facts and preferences **asynchronously**, usually within a few minutes. The stack declares these namespace templates, so for this actor the records live under `/facts/<actor_id>/` and `/preferences/<actor_id>/`. If nothing shows up yet, wait and run the cell again.

In [ ]:
time.sleep(5)
for label, namespace in (("facts", f"/facts/{ACTOR_ID}/"), ("preferences", f"/preferences/{ACTOR_ID}/")):
    try:
        result = agentcore.retrieve_memory_records(
            memoryId=MEMORY_ID,
            namespace=namespace,
            searchCriteria={"searchQuery": "user name, country and interests", "topK": 10},
        )
        records = result.get("memoryRecordSummaries", [])
        print(f"{label}: {len(records)} record(s)")
        for record in records:
            print("  -", str(record.get("content", {}).get("text", record))[:200])
    except Exception as error:  # noqa: BLE001 - notebook diagnostics
        print(f"{label}: could not query memory ({type(error).__name__}: {error})")

## Clean up

Nothing is created outside the stack, except the video you may have uploaded under `video/`. The bucket applies a 30 day expiration to that prefix.